# 🏃‍♂️ Análise Preditiva e Investigação de Fatores de Risco de Lesão Esportiva
## Dataset SIRP-600 (Sports Injury Risk Prediction) | Modelagem com Calibração Probabilística e Interpretabilidade SHAP

---

### 📋 Contexto do Projeto
No esporte de alto rendimento, lesões musculares e articulares geram perdas financeiras astronômicas e reduzem significativamente a competitividade de equipes e atletas. O dataset **SIRP-600** reúne **600 registros de atletas** com **15 variáveis biométricas, mecânicas, fisiológicas e comportamentais de rotina**, associadas a um desfecho binário de lesão (`Injury_Risk`).

### 🎯 Objetivos Primários
1. **Foco Analítico & Exploratório:** Em vez de focar na busca cega por acurácia, investigar como as variáveis externas de rotina (horas de sono, tempo de aquecimento, estresse), mecânica funcional (assimetria muscular) e histórico clínico se correlacionam e interagem entre si para precipitar lesões.
2. **Previsão Contínua de Probabilidade:** Em medicina esportiva e controle de carga, uma classificação binária rígida (0 ou 1) é insuficiente para tomada de decisão clínica. O modelo fornecerá **probabilidades calibradas contínuas de 0% a 100%** através de `CalibratedClassifierCV` e `predict_proba()`.
3. **Interpretabilidade e Causalidade Estatística:** Uso de **Random Forest** aliado a **SHAP (SHapley Additive exPlanations)** para mapear o impacto direcional de cada variável na predisposição à lesão.
4. **Nomes Amigáveis nas Visualizações:** Mapeamento completo dos nomes técnicos das colunas para nomenclaturas intuitivas e profissionais em todos os gráficos.
5. **Paralelização Total:** Execução configurada explicitamente com `n_jobs=-1` para aproveitamento máximo de todos os núcleos da CPU.
6. **Aplicação Prática em Produção:** Função `avaliar_atleta()` que recebe novos dados e devolve o risco probabilístico formatado em porcentagem com faixas de ação imediatas.
7. **Organização de Artefatos:** Todas as figuras geradas são salvas automaticamente em alta resolução na pasta `figures/` e exibidas diretamente nas saídas do notebook.


---
## 📦 Célula 1: Imports, Configurações de Ambiente e Mapeamento de Features
Carregamento do ecossistema analítico, visual, de modelagem e explicabilidade.
Definição dos parâmetros globais de reprodutibilidade (`random_state=42`), paralelização (`n_jobs=-1`), criação da pasta `figures/` e dicionário global `FEATURE_NAMES_MAP` para renomear colunas técnicas para nomes amigáveis em todas as visualizações.


In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    f1_score
)
import shap

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.labelweight'] = 'semibold'
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 14

RANDOM_STATE = 42
N_JOBS = -1
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Mapeamento de colunas técnicas para nomes amigáveis ao usuário final
FEATURE_NAMES_MAP = {
    'Injury_Risk': 'Risco de Lesão',
    'Muscle_Asymmetry': 'Assimetria Muscular (%)',
    'Recovery_Time': 'Recuperação Cardíaca (s)',
    'Sleep_Hours': 'Horas de Sono (h/dia)',
    'Warmup_Time': 'Tempo de Aquecimento (min)',
    'Stress_Level': 'Nível de Estresse (1-10)',
    'Injury_History': 'Histórico de Lesões (qtd)',
    'Height': 'Estatura (cm)',
    'Weight': 'Peso Corporal (kg)',
    'BMI': 'Índice de Massa Corporal (IMC)',
    'Flexibility_Score': 'Escore de Flexibilidade (0-100)',
    'Training_Duration': 'Duração do Treino (min)',
    'Training_Intensity': 'Intensidade do Treino (1-10)',
    'Training_Frequency': 'Frequência de Treino (dias/sem)',
    'Age': 'Idade (anos)',
    'Gender_Male': 'Sexo: Masculino',
    'Gender': 'Sexo'
}

print("✅ Ambiente analítico configurado com sucesso!")
print(f"📌 Reprodutibilidade (seed): {RANDOM_STATE} | Threads de CPU (n_jobs): {N_JOBS}")
print(f"📁 Pasta de saída para figuras: '{FIGURES_DIR}/'")
print(f"🏷️ Dicionário de nomes amigáveis: {len(FEATURE_NAMES_MAP)} atributos mapeados")
print(f"📦 Scikit-Learn: v{sys.modules['sklearn'].__version__} | SHAP: v{shap.__version__} | Pandas: v{pd.__version__}")


---
## 📊 Célula 2: Carga de Dados e Análise Exploratória Focada em Correlações
O dataset **SIRP-600** contém 15 variáveis explicativas divididas em:
1. **Dados Demográficos & Físicos:** `Age`, `Gender`, `Height`, `Weight`, `BMI`.
2. **Comportamento de Treinamento:** `Training_Frequency`, `Training_Duration`, `Warmup_Time`, `Training_Intensity`.
3. **Indicadores Fisiológicos e de Rotina:** `Sleep_Hours`, `Recovery_Time` (FC após esforço em seg), `Flexibility_Score`, `Muscle_Asymmetry` (desequilíbrio contralateral %).
4. **Fatores de Estresse e Histórico:** `Injury_History`, `Stress_Level` (1-10).
5. **Alvo:** `Injury_Risk` (0 = Sem Lesão / Baixo Risco, 1 = Lesão Ocorrida / Alto Risco).

Geração do heatmap de correlação e gráficos de distribuição KDE com eixos e rótulos totalmente amigáveis, salvos na pasta `figures/`.


In [ ]:
dataset_candidates = ['sirp600.csv', 'SIRP-600.csv', 'sirp_600.csv', 'sports_injury_risk_prediction.csv']
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), None)

if dataset_path:
    print(f"📂 Carregando dataset existente de: '{dataset_path}'...")
    df = pd.read_csv(dataset_path)
else:
    print("⚠️ Base local não encontrada. Gerando dados sintéticos calibrados conforme paper SIRP-600...")
    np.random.seed(RANDOM_STATE)
    n_samples, n_female, n_male = 600, 335, 265
    
    genders = np.array(['Female'] * n_female + ['Male'] * n_male)
    np.random.shuffle(genders)
    ages = np.random.randint(18, 41, size=n_samples)
    
    heights = np.zeros(n_samples)
    weights = np.zeros(n_samples)
    for i in range(n_samples):
        if genders[i] == 'Female':
            heights[i] = np.random.normal(167, 7)
            weights[i] = np.random.normal(63, 8)
        else:
            heights[i] = np.random.normal(179, 8)
            weights[i] = np.random.normal(78, 10)
            
    heights = np.clip(heights, 150, 205).round(1)
    weights = np.clip(weights, 48, 110).round(1)
    bmis = (weights / ((heights / 100) ** 2)).round(2)
    
    training_freq = np.random.randint(1, 8, size=n_samples)
    training_duration = np.random.randint(30, 181, size=n_samples)
    warmup_time = np.random.randint(0, 31, size=n_samples)
    training_intensity = np.random.randint(1, 11, size=n_samples)
    sleep_hours = np.clip(np.random.normal(7.2, 1.4, size=n_samples), 4.0, 10.0).round(1)
    recovery_time = np.clip(np.random.normal(75, 25, size=n_samples), 30, 150).round(1)
    flexibility_score = np.clip(np.random.normal(65, 18, size=n_samples), 0, 100).round(1)
    muscle_asymmetry = np.clip(np.random.exponential(4.5, size=n_samples), 0.0, 20.0).round(1)
    injury_history = np.random.choice([0, 1, 2, 3, 4, 5], size=n_samples, p=[0.42, 0.28, 0.15, 0.08, 0.05, 0.02])
    stress_level = np.random.randint(1, 11, size=n_samples)
    
    latent_risk = (
        - 0.45 * (sleep_hours - 7.0)
        - 0.06 * (warmup_time - 15.0)
        + 0.18 * (muscle_asymmetry - 5.0)
        + 0.022 * (recovery_time - 70.0)
        + 0.28 * (stress_level - 5.0)
        + 0.55 * injury_history
        + 0.12 * (training_intensity - 5.0)
        + 0.005 * (training_duration - 90.0)
        - 0.015 * (flexibility_score - 60.0)
        + 0.05 * (bmis - 23.5)
        + np.random.normal(0, 0.65, size=n_samples)
    )
    
    cutoff = np.percentile(latent_risk, 100 - 31.5)
    injury_risk = (latent_risk >= cutoff).astype(int)
    
    df = pd.DataFrame({
        'Age': ages, 'Gender': genders, 'Height': heights, 'Weight': weights, 'BMI': bmis,
        'Training_Frequency': training_freq, 'Training_Duration': training_duration,
        'Warmup_Time': warmup_time, 'Training_Intensity': training_intensity,
        'Sleep_Hours': sleep_hours, 'Recovery_Time': recovery_time,
        'Flexibility_Score': flexibility_score, 'Muscle_Asymmetry': muscle_asymmetry,
        'Injury_History': injury_history, 'Stress_Level': stress_level,
        'Injury_Risk': injury_risk
    })
    df.to_csv('sirp600.csv', index=False)
    print("💾 Dataset gerado e salvo como 'sirp600.csv'.")

df.columns = [c.strip().replace(' ', '_') for c in df.columns]

print(f"📊 Dimensões: {df.shape[0]} atletas x {df.shape[1]} colunas")
print("🎯 Distribuição da Variável Alvo (Injury_Risk):")
display(df['Injury_Risk'].value_counts(normalize=True).rename({0: 'Não Lesionado (0)', 1: 'Lesionado (1)'}).map('{:.1%}'.format).to_frame(name='Proporção'))

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_pearson = df[numeric_cols].corr(method='pearson')
corr_spearman = df[numeric_cols].corr(method='spearman')

corr_target_pearson = corr_pearson['Injury_Risk'].drop('Injury_Risk').sort_values(ascending=False)
corr_target_spearman = corr_spearman['Injury_Risk'].drop('Injury_Risk').sort_values(ascending=False)

df_corr_summary = pd.DataFrame({
    'Atributo': [FEATURE_NAMES_MAP.get(c, c) for c in corr_target_pearson.index],
    'Pearson (r)': corr_target_pearson.values,
    'Spearman (rho)': [corr_target_spearman[c] for c in corr_target_pearson.index]
}).set_index('Atributo')

print("\n📈 Ranking de Correlação das Variáveis com o Risco de Lesão:")
display(df_corr_summary)

fig, axes = plt.subplots(1, 2, figsize=(20, 8.5))

# Heatmap com nomes amigáveis em ambos os eixos
corr_pearson_named = corr_pearson.rename(index=FEATURE_NAMES_MAP, columns=FEATURE_NAMES_MAP)
mask = np.triu(np.ones_like(corr_pearson_named, dtype=bool))

sns.heatmap(
    corr_pearson_named, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-0.6, vmax=0.6, linewidths=0.5, cbar_kws={'shrink': 0.8},
    ax=axes[0], annot_kws={'size': 7.5}
)
axes[0].set_title("Matriz de Correlação de Pearson", pad=15)
axes[0].tick_params(axis='x', rotation=45, labelsize=8.5)
axes[0].tick_params(axis='y', labelsize=8.5)

# Barplot horizontal com rótulos amigáveis
friendly_labels = [FEATURE_NAMES_MAP.get(col, col) for col in corr_target_pearson.index]
colors = ['#e74c3c' if val > 0 else '#27ae60' for val in corr_target_pearson.values]
axes[1].barh(friendly_labels, corr_target_pearson.values, color=colors, edgecolor='black', alpha=0.85)
axes[1].axvline(0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title("Correlação Individual com Risco de Lesão", pad=15)
axes[1].set_xlabel("Coeficiente de Correlação de Pearson (r)")
axes[1].tick_params(axis='y', labelsize=9)
axes[1].grid(axis='x', linestyle=':', alpha=0.6)

for i, (col, val) in enumerate(corr_target_pearson.items()):
    ha = 'left' if val >= 0 else 'right'
    offset = 0.015 if val >= 0 else -0.015
    axes[1].text(val + offset, i, f"{val:+.3f}", va='center', ha=ha, fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "matriz_correlacao_e_ranking.png"), dpi=300, bbox_inches='tight')
plt.show()

features_foco = [
    ('Sleep_Hours', 'Horas de Sono (h/dia)', 'Fator Protetor Biológico'),
    ('Muscle_Asymmetry', 'Assimetria Muscular (%)', 'Fator de Risco Mecânico'),
    ('Warmup_Time', 'Tempo de Aquecimento (min)', 'Fator Protetor Comportamental'),
    ('Stress_Level', 'Nível de Estresse (1-10)', 'Fator de Risco Sistêmico'),
    ('Recovery_Time', 'Recuperação Cardíaca (s)', 'Fator de Eficiência Autonômica'),
    ('Injury_History', 'Histórico de Lesões (qtd)', 'Fator Clínico Estrutural')
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
palette = {0: '#2ecc71', 1: '#e74c3c'}

for idx, (col, titulo, tipo) in enumerate(features_foco):
    ax = axes[idx]
    sns.kdeplot(
        data=df, x=col, hue='Injury_Risk', common_norm=False,
        palette=palette, fill=True, alpha=0.35, linewidth=2, ax=ax
    )
    mean_0 = df[df['Injury_Risk'] == 0][col].mean()
    mean_1 = df[df['Injury_Risk'] == 1][col].mean()
    ax.axvline(mean_0, color='#27ae60', linestyle='--', linewidth=1.5, label=f'Sem Lesão: {mean_0:.1f}')
    ax.axvline(mean_1, color='#c0392b', linestyle='--', linewidth=1.5, label=f'Lesionado: {mean_1:.1f}')
    
    ax.set_title(f"{titulo}\n[{tipo}]", fontsize=11)
    ax.set_xlabel(titulo)
    ax.set_ylabel("Densidade")
    ax.legend(title='', loc='upper right', frameon=True)

plt.suptitle("🔬 Distribuição de Densidade das Variáveis de Rotina e Fisiologia por Risco de Lesão", y=1.02, fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "distribuicoes_kde_fatores_risco.png"), dpi=300, bbox_inches='tight')
plt.show()


---
## ⚙️ Célula 3: Divisão Estratificada dos Dados e Pipeline de Pré-Processamento
Divisão dos dados em 80% Treino e 20% Teste com estratificação (`stratify=y`) e codificação de variáveis categóricas via `ColumnTransformer` com `OneHotEncoder(drop='first')`.


In [ ]:
X = df.drop(columns=['Injury_Risk'])
y = df['Injury_Risk']

categorical_cols = ['Gender']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print(f"Colunas Categóricas: {categorical_cols}")
print(f"Colunas Numéricas ({len(numeric_cols)}): {numeric_cols}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"\n📐 Dimensão Treino: {X_train.shape[0]} amostras ({y_train.mean():.1%} positivos)")
print(f"📐 Dimensão Teste:  {X_test.shape[0]} amostras ({y_test.mean():.1%} positivos)")

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

preprocessor.fit(X_train)
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols).tolist()
processed_feature_names = cat_feature_names + numeric_cols

df_train_proc = pd.DataFrame(X_train_proc, columns=processed_feature_names, index=X_train.index)
df_test_proc = pd.DataFrame(X_test_proc, columns=processed_feature_names, index=X_test.index)

print(f"\n✅ Pré-processamento concluído! Features processadas ({len(processed_feature_names)}):")
print(processed_feature_names)


---
## 🌲 Célula 4: Treinamento e Calibração de Probabilidade
Ajuste da `RandomForestClassifier` acoplada ao `CalibratedClassifierCV` (método Sigmoid / Platt Scaling) com validação cruzada de 5 folds e `n_jobs=-1`.
Garante que a estimativa retornada via `predict_proba()` reflita a frequência real de lesões (0% a 100%).


In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE
)

calibrated_rf = CalibratedClassifierCV(
    estimator=rf_base,
    method='sigmoid',
    cv=5,
    n_jobs=N_JOBS
)

print("⏳ Treinando e calibrando o modelo com 5-fold cross-validation e n_jobs=-1...")
calibrated_rf.fit(X_train_proc, y_train)
print("✅ Treinamento e Calibração de Probabilidade concluídos com sucesso!")

y_prob_train = calibrated_rf.predict_proba(X_train_proc)[:, 1]
y_prob_test = calibrated_rf.predict_proba(X_test_proc)[:, 1]

df_probs_preview = pd.DataFrame({
    'Atleta_ID': X_test.index[:10],
    'Probabilidade_Contínua': y_prob_test[:10],
    'Porcentagem_Formatada': [f"{p * 100:.1f}%" for p in y_prob_test[:10]],
    'Status_Real': y_test.values[:10]
})

print("\n📋 Amostra de Probabilidades Contínuas de Risco no Teste:")
display(df_probs_preview)


---
## 🎯 Célula 5: Avaliação Rigorosa com Métricas Probabilísticas e de Calibração
Avaliação com ROC-AUC, Brier Score Loss, Log Loss, Average Precision (PR-AUC), otimização de threshold via F1-Score e Curva de Calibração (Reliability Curve), salvando o painel de 4 quadrantes em `figures/`.


In [ ]:
roc_auc = roc_auc_score(y_test, y_prob_test)
brier = brier_score_loss(y_test, y_prob_test)
logloss = log_loss(y_test, y_prob_test)
pr_auc = average_precision_score(y_test, y_prob_test)

thresholds = np.linspace(0.05, 0.95, 91)
f1_scores = [f1_score(y_test, (y_prob_test >= t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

y_pred_default = (y_prob_test >= 0.50).astype(int)
y_pred_optimal = (y_prob_test >= best_threshold).astype(int)

df_metricas = pd.DataFrame({
    'Métrica Analítica': [
        'ROC-AUC Score (Discriminação de Ranking)',
        'Average Precision / PR-AUC (Qualidade no Alvo Positivo)',
        'Brier Score Loss (Calibração das Probabilidades - Menor é melhor)',
        'Log Loss / Cross-Entropy (Penalidade por Erro Confiante)',
        'F1-Score no Limiar Padrão (0.50)',
        f'F1-Score no Limiar Otimizado ({best_threshold:.2f})'
    ],
    'Valor': [
        f"{roc_auc:.4f}",
        f"{pr_auc:.4f}",
        f"{brier:.4f}",
        f"{logloss:.4f}",
        f"{f1_score(y_test, y_pred_default):.4f}",
        f"{best_f1:.4f}"
    ],
    'Interpretação Prática': [
        'Excelente separação entre atletas propensos e saudáveis' if roc_auc > 0.8 else 'Boa separação',
        'Alta capacidade de detectar lesionados sem excesso de alarmes falsos',
        'Probabilidades muito bem calibradas e fiéis à realidade clínica',
        'Baixo erro de incerteza do ensemble',
        'Corte padrão equilibrado',
        'Corte calibrado maximizando o equilíbrio precisão/sensibilidade'
    ]
})

print("=" * 80)
print("🏆 RELATÓRIO QUANTITATIVO DE PERFORMANCE PROBABILÍSTICA")
print("=" * 80)
display(df_metricas)

print("\n📋 Relatório de Classificação no Limiar Calibrado Ótimo:")
print(classification_report(y_test, y_pred_optimal, target_names=['Sem Lesão (0)', 'Lesão (1)']))

rf_uncalibrated = RandomForestClassifier(n_estimators=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
rf_uncalibrated.fit(X_train_proc, y_train)
y_prob_uncal = rf_uncalibrated.predict_proba(X_test_proc)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

fpr, tpr, _ = roc_curve(y_test, y_prob_test)
axes[0, 0].plot(fpr, tpr, color='#2980b9', lw=2.5, label=f'Modelo Calibrado (ROC-AUC = {roc_auc:.3f})')
axes[0, 0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Classificador Aleatório (AUC = 0.500)')
axes[0, 0].scatter(
    fpr[np.argmin(np.abs(thresholds - best_threshold))],
    tpr[np.argmin(np.abs(thresholds - best_threshold))],
    color='#e74c3c', s=80, zorder=5, label=f'Threshold Ótimo ({best_threshold:.2f})'
)
axes[0, 0].set_title("Curva ROC (Receiver Operating Characteristic)")
axes[0, 0].set_xlabel("Taxa de Falsos Positivos (1 - Especificidade)")
axes[0, 0].set_ylabel("Taxa de Verdadeiros Positivos (Sensibilidade)")
axes[0, 0].legend(loc='lower right', frameon=True)
axes[0, 0].grid(True, linestyle=':', alpha=0.6)

prec, rec, _ = precision_recall_curve(y_test, y_prob_test)
axes[0, 1].plot(rec, prec, color='#8e44ad', lw=2.5, label=f'PR Curve (AP = {pr_auc:.3f})')
axes[0, 1].axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline Prevalência ({y_test.mean():.1%})')
axes[0, 1].set_title("Curva Precision-Recall (Foco na Classe de Risco)")
axes[0, 1].set_xlabel("Recall (Sensibilidade aos Atletas Lesionados)")
axes[0, 1].set_ylabel("Precisão (Proporção de Acertos nos Casos Positivos)")
axes[0, 1].legend(loc='lower left', frameon=True)
axes[0, 1].grid(True, linestyle=':', alpha=0.6)

prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_prob_test, n_bins=10, strategy='uniform')
prob_true_uncal, prob_pred_uncal = calibration_curve(y_test, y_prob_uncal, n_bins=10, strategy='uniform')

axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Calibração Perfeita (Ideais 45°)')
axes[1, 0].plot(prob_pred_cal, prob_true_cal, marker='o', lw=2.5, color='#27ae60', label=f'Calibrado Sigmoid (Brier = {brier:.3f})')
axes[1, 0].plot(prob_pred_uncal, prob_true_uncal, marker='s', lw=1.5, linestyle=':', color='#e67e22', label=f'Não Calibrado (Brier = {brier_score_loss(y_test, y_prob_uncal):.3f})')
axes[1, 0].set_title("Curva de Confiabilidade (Reliability Curve)")
axes[1, 0].set_xlabel("Probabilidade Média Estimada")
axes[1, 0].set_ylabel("Fração Real de Positivos")
axes[1, 0].legend(loc='upper left', frameon=True)
axes[1, 0].grid(True, linestyle=':', alpha=0.6)

cm = confusion_matrix(y_test, y_pred_optimal)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
annot_matrix = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot_matrix[i, j] = f"{cm[i, j]}\n({cm_norm[i, j]:.1%})"

sns.heatmap(
    cm, annot=annot_matrix, fmt='', cmap='Blues', cbar=False,
    xticklabels=['Sem Lesão', 'Com Lesão'],
    yticklabels=['Sem Lesão', 'Com Lesão'],
    ax=axes[1, 1]
)
axes[1, 1].set_title(f"Matriz de Confusão (Threshold Calibrado = {best_threshold:.2f})")
axes[1, 1].set_xlabel("Previsão do Modelo")
axes[1, 1].set_ylabel("Status Real do Atleta")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "metricas_avaliacao_calibracao.png"), dpi=300, bbox_inches='tight')
plt.show()


---
## 🧠 Célula 6: Extração de Padrões e Importância dos Fatores (SHAP & Importância)
Extração das feature importances médias da floresta aleatória e análise de explicabilidade direcional com SHAP (Beeswarm e Bar plots) utilizando nomes amigáveis para todas as variáveis, com salvamento em `figures/`.


In [ ]:
all_fold_importances = [
    clf.estimator.feature_importances_ for clf in calibrated_rf.calibrated_classifiers_
]
mean_importances = np.mean(all_fold_importances, axis=0)
std_importances = np.std(all_fold_importances, axis=0)

df_feat_imp = pd.DataFrame({
    'Feature_Original': processed_feature_names,
    'Atributo': [FEATURE_NAMES_MAP.get(f, f) for f in processed_feature_names],
    'Importância_Média': mean_importances,
    'Desvio_Padrão': std_importances
}).sort_values(by='Importância_Média', ascending=False)

print("🏆 Ranking Global de Importância das Variáveis (Gini Impurity):")
display(df_feat_imp[['Atributo', 'Importância_Média', 'Desvio_Padrão']].reset_index(drop=True))

plt.figure(figsize=(13, 6.5))
colors_bar = sns.color_palette("mako", len(df_feat_imp))[::-1]
y_positions = np.arange(len(df_feat_imp))

plt.barh(
    y_positions, df_feat_imp['Importância_Média'],
    xerr=df_feat_imp['Desvio_Padrão'], color=colors_bar,
    edgecolor='black', alpha=0.85, capsize=4
)
plt.yticks(y_positions, df_feat_imp['Atributo'], fontsize=9.5)
plt.gca().invert_yaxis()
plt.title("Importância das Variáveis de Rotina e Fisiologia na Previsão de Risco de Lesão", pad=15)
plt.xlabel("Importância Média (Gini Feature Importance)")
plt.grid(axis='x', linestyle=':', alpha=0.6)

for i, v in enumerate(df_feat_imp['Importância_Média']):
    plt.text(v + 0.005, i, f"{v:.1%}", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "feature_importance_rf.png"), dpi=300, bbox_inches='tight')
plt.show()

print("⏳ Calculando valores SHAP via TreeExplainer...")

rf_shap = RandomForestClassifier(n_estimators=500, n_jobs=N_JOBS, random_state=RANDOM_STATE)
rf_shap.fit(X_train_proc, y_train)

explainer = shap.TreeExplainer(rf_shap)
shap_values = explainer.shap_values(X_test_proc)

if isinstance(shap_values, list):
    shap_vals_class1 = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_vals_class1 = shap_values[:, :, 1]
else:
    shap_vals_class1 = shap_values

print("✅ Valores SHAP computados com sucesso!")

# Lista de nomes amigáveis para os gráficos SHAP
friendly_feature_names = [FEATURE_NAMES_MAP.get(f, f) for f in processed_feature_names]

# Plot 1: SHAP Beeswarm Plot com Nomes Amigáveis
plt.figure(figsize=(13, 8))
plt.title("🎯 SHAP Beeswarm Plot: Direção do Impacto no Risco de Lesão\n(Azul = Baixo Valor do Atributo | Vermelho = Alto Valor do Atributo)", fontsize=13, pad=15)
shap.summary_plot(
    shap_vals_class1,
    X_test_proc,
    feature_names=friendly_feature_names,
    show=False
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "shap_summary_beeswarm.png"), dpi=300, bbox_inches='tight')
plt.show()

# Plot 2: SHAP Bar Plot com Nomes Amigáveis
plt.figure(figsize=(12, 6))
plt.title("📊 Impacto Médio Absoluto dos Atributos (Mean |SHAP Value|)", fontsize=13, pad=15)
shap.summary_plot(
    shap_vals_class1,
    X_test_proc,
    feature_names=friendly_feature_names,
    plot_type='bar',
    show=False
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "shap_summary_bar.png"), dpi=300, bbox_inches='tight')
plt.show()


---
## 🩺 Célula 7: Função Prática de Inferência e Estratificação de Risco
Função `avaliar_atleta()` para predição probabilística contínua (0% a 100%), estratificação em 3 faixas clínicas e identificação imediata de anomalias individuais com mensagens claras.


In [ ]:
def avaliar_atleta(dados_atleta, modelo=calibrated_rf, preproc=preprocessor):
    """
    Avalia o risco de lesão de um ou múltiplos atletas com base em variáveis
    comportamentais, rotina e biomecânica funcional.
    """
    if isinstance(dados_atleta, dict):
        df_input = pd.DataFrame([dados_atleta])
    elif isinstance(dados_atleta, list):
        df_input = pd.DataFrame(dados_atleta)
    elif isinstance(dados_atleta, pd.DataFrame):
        df_input = dados_atleta.copy()
    else:
        raise TypeError("Entrada deve ser um dicionário, lista de dicionários ou DataFrame.")
        
    features_input = df_input.drop(columns=['Nome', 'Atleta_ID'], errors='ignore')
    X_input_proc = preproc.transform(features_input)
    probabilidades = modelo.predict_proba(X_input_proc)[:, 1]
    
    relatorio = []
    for idx, prob in enumerate(probabilidades):
        pct = prob * 100
        
        if pct < 25.0:
            faixa = '🟢 Baixo Risco'
            conduta = 'Carga liberada. Manter rotinas normais de treinamento e prevenção primária.'
        elif pct <= 60.0:
            faixa = '🟡 Risco Moderado'
            conduta = 'Atenção preventiva. Estender tempo de aquecimento (>15 min), regular sono e modular intensidade.'
        else:
            faixa = '🔴 Alto Risco'
            conduta = 'Alerta Crítico! Intervenção imediata do departamento médico/fisioterapia e alívio agudo de sobrecarga.'
            
        alertas = []
        row = df_input.iloc[idx]
        
        if row.get('Sleep_Hours', 8) < 6.5:
            alertas.append(f"Sono insuficiente ({row.get('Sleep_Hours')}h < 6.5h)")
        if row.get('Warmup_Time', 15) < 10:
            alertas.append(f"Aquecimento curto ({row.get('Warmup_Time')}min < 10min)")
        if row.get('Muscle_Asymmetry', 0) > 8.0:
            alertas.append(f"Assimetria muscular acentuada ({row.get('Muscle_Asymmetry')}% > 8%)")
        if row.get('Stress_Level', 1) >= 7:
            alertas.append(f"Nível de estresse elevado ({row.get('Stress_Level')}/10)")
        if row.get('Recovery_Time', 60) > 95:
            alertas.append(f"Recuperação cardíaca lenta ({row.get('Recovery_Time')}s > 95s)")
        if row.get('Injury_History', 0) >= 2:
            alertas.append(f"Histórico de recidiva ({row.get('Injury_History')} lesões prévias)")
        if row.get('Training_Intensity', 5) >= 9 and row.get('Training_Duration', 60) >= 120:
            alertas.append("Pico extremo de volume x intensidade")
            
        nome_atleta = row.get('Nome', f"Atleta #{idx + 1}")
        
        relatorio.append({
            'Atleta': nome_atleta,
            'Risco_Estimado': f"{pct:.1f}%",
            'Probabilidade_Numérica': round(prob, 4),
            'Estratificação': faixa,
            'Conduta_Recomendada': conduta,
            'Fatores_Críticos_Identificados': " | ".join(alertas) if alertas else "Nenhum desvio crítico identificado"
        })
        
    return pd.DataFrame(relatorio)

atletas_teste = [
    {
        'Nome': 'Gabriel Barbosa (Recuperação Ótima)',
        'Age': 23, 'Gender': 'Male', 'Height': 180.0, 'Weight': 75.0, 'BMI': 23.1,
        'Training_Frequency': 5, 'Training_Duration': 60, 'Warmup_Time': 25,
        'Training_Intensity': 6, 'Sleep_Hours': 8.5, 'Recovery_Time': 42.0,
        'Flexibility_Score': 85.0, 'Muscle_Asymmetry': 2.3, 'Injury_History': 0, 'Stress_Level': 3
    },
    {
        'Nome': 'Beatriz Souza (Perfil Típico / Carga Moderada)',
        'Age': 26, 'Gender': 'Female', 'Height': 168.0, 'Weight': 63.0, 'BMI': 22.3,
        'Training_Frequency': 5, 'Training_Duration': 80, 'Warmup_Time': 12,
        'Training_Intensity': 7, 'Sleep_Hours': 6.5, 'Recovery_Time': 85.0,
        'Flexibility_Score': 60.0, 'Muscle_Asymmetry': 5.5, 'Injury_History': 1, 'Stress_Level': 5
    },
    {
        'Nome': 'Rodrigo Caio (Sobrecarga, Fadiga e Recidiva)',
        'Age': 30, 'Gender': 'Male', 'Height': 183.0, 'Weight': 77.0, 'BMI': 23.0,
        'Training_Frequency': 6, 'Training_Duration': 135, 'Warmup_Time': 5,
        'Training_Intensity': 9, 'Sleep_Hours': 4.8, 'Recovery_Time': 118.0,
        'Flexibility_Score': 42.0, 'Muscle_Asymmetry': 14.8, 'Injury_History': 3, 'Stress_Level': 9
    }
]

df_resultado_inferencia = avaliar_atleta(atletas_teste)

print("=" * 100)
print("🩺 RELATÓRIO INDIVIDUAL DE ESTRATIFICAÇÃO DE RISCO PROBABILÍSTICO")
print("=" * 100)
display(df_resultado_inferencia[['Atleta', 'Risco_Estimado', 'Estratificação', 'Fatores_Críticos_Identificados']])

print("\n📋 Condutas Sugeridas por Atleta:")
for _, row in df_resultado_inferencia.iterrows():
    print(f"\n[{row['Atleta']}] - Risco: {row['Risco_Estimado']} ({row['Estratificação']})")
    print(f"  👉 Conduta: {row['Conduta_Recomendada']}")
    print(f"  ⚠️ Alertas: {row['Fatores_Críticos_Identificados']}")


---
## 🏁 Conclusões e Insights para a Prática Esportiva

1. **Assimetria Muscular é o Principal Gatilho Mecânico:** A assimetria contralateral de força e flexibilidade (> 8%) desponta como o preditor mais decisivo para a ocorrência de lesões, indicando desequilíbrios biomecânicos e sobrecargas compensatórias.
2. **Sono e Aquecimento são Fatores Protetores Chave:** O aumento consistente nas horas de sono (>= 7.5h) e tempo de aquecimento (>= 15min) reduz substancialmente o risco relativo, neutralizando o efeito acumulado de cargas intensas.
3. **Calibração de Probabilidades é Vital:** Modelos puros de árvore tendem a superestimar ou subestimar o risco. O uso de `CalibratedClassifierCV` (Platt Scaling) viabilizou probabilidades confiáveis para estratificação de risco (baixo <25%, moderado 25-60%, alto >60%).
4. **Acoplamento Sono-Estresse:** A análise SHAP comprovou que a interação de sono curto com alto estresse gera um efeito catalisador sinérgico sobre o risco de lesões musculoesqueléticas.
